# Week07 - Capstone project - Hyperparameter tuning

## Strategy
For Week 7, the query policy emphasises **hyperparameter tuning** of the optimiser itself.

Instead of treating the surrogate model as fixed, this notebook searches over a small grid of settings for:
- surrogate type (`GP` or shallow `MLP ensemble`)
- exploration strength (`beta`)
- local step size
- grid rounding resolution

For each function, the tuned configuration is chosen using a simple leave-one-out style validation score on the current data. The final candidate is then selected from a **human-clean local candidate pool** so the output remains consistent with the historical submission style.


In [1]:
import itertools
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.model_selection import KFold
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


/Users/sundarid/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
data_dir = Path('data')
history = pd.read_csv(data_dir / 'weekly_results/Week07.csv')
history.tail(8)


,week,function,d,x1,x2,x3,x4,x5,x6,x7,x8,y
40,6,1,2,0.700,0.60,NaN,NaN,NaN,NaN,NaN,NaN,-0.000061
41,6,2,2,0.500,0.70,NaN,NaN,NaN,NaN,NaN,NaN,0.609311
42,6,3,3,0.350,0.55,0.45,NaN,NaN,NaN,NaN,NaN,-0.006480
43,6,4,4,0.400,0.40,0.35,0.45,NaN,NaN,NaN,NaN,-0.016838
44,6,5,4,0.940,0.08,0.97,0.96,NaN,NaN,NaN,NaN,2917.697037
45,6,6,5,0.600,0.26,0.72,0.90,0.09,NaN,NaN,NaN,-0.524374
46,6,7,6,0.025,0.26,0.36,0.26,0.48,0.76,NaN,NaN,1.611028
47,6,8,8,0.400,0.10,0.38,0.07,0.80,0.10,0.11,0.09,9.421390


In [3]:
def load_initial(function_id):
    X0 = np.load(data_dir / f'initial_data/F{function_id}_initial_inputs.npy')
    y0 = np.load(data_dir / f'initial_data/F{function_id}_initial_outputs.npy')
    return X0, y0

def load_combined(function_id, history_df):
    X0, y0 = load_initial(function_id)
    rows = history_df[history_df['function'] == function_id].sort_values('week')
    d = int(rows['d'].iloc[0])
    Xw = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)
    yw = rows['y'].to_numpy(dtype=float)
    X = np.vstack([X0, Xw])
    y = np.concatenate([y0, yw])
    return X, y, d, rows

def round_grid(x, grid):
    return np.clip(np.round(x / grid) * grid, 0.0, 1.0)

def format_point(x):
    return '-'.join(f'{float(v):.6f}' for v in x)


In [4]:
def build_gp(seed, d):
    kernel = (
        ConstantKernel(1.0, (0.1, 10.0))
        * RBF(length_scale=np.ones(d), length_scale_bounds=(0.05, 2.0))
        + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-2))
    )
    return GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=seed,
        n_restarts_optimizer=1,
    )

def mlp_ensemble_predict(X_train, y_train, X_test, seeds):
    preds = []
    for seed in seeds:
        model = make_pipeline(
            StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=(16, 8),
                activation='relu',
                solver='adam',
                alpha=1e-3,
                learning_rate_init=1e-2,
                max_iter=2500,
                random_state=seed,
            )
        )
        model.fit(X_train, y_train)
        preds.append(model.predict(X_test))
    preds = np.vstack(preds)
    return preds.mean(axis=0), preds.std(axis=0)


In [5]:
def build_candidates(best_x, latest_x, second_x, d, step, grid):
    candidates = []
    anchors = [best_x, latest_x, second_x, (best_x + latest_x) / 2.0, (best_x + second_x) / 2.0]
    for a in anchors:
        candidates.append(round_grid(a, grid))

    for i in range(d):
        for sign in (-1.0, 1.0):
            x = best_x.copy()
            x[i] += sign * step
            candidates.append(round_grid(x, grid))

    for i in range(min(d, 3)):
        x = best_x.copy()
        x[i] += step
        candidates.append(round_grid(x, grid))
        x = best_x.copy()
        x[i] -= step
        candidates.append(round_grid(x, grid))

    return np.unique(np.array(candidates), axis=0)


In [6]:
def cv_score_surrogate(X, y, model_type, seed):
    kf = KFold(n_splits=min(4, len(X)), shuffle=True, random_state=seed)
    scores = []
    for train_idx, test_idx in kf.split(X):
        Xtr, Xte = X[train_idx], X[test_idx]
        ytr, yte = y[train_idx], y[test_idx]
        if model_type == 'gp':
            model = build_gp(seed, X.shape[1])
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore', category=ConvergenceWarning)
                model.fit(Xtr, ytr)
            pred = model.predict(Xte)
        else:
            pred, _ = mlp_ensemble_predict(Xtr, ytr, Xte, seeds=[seed, seed+1, seed+2])
        rmse = float(np.sqrt(np.mean((pred - yte) ** 2)))
        scores.append(rmse)
    return float(np.mean(scores))


In [7]:
def suggest_week7_point(function_id, history_df, seed=500):
    X, y, d, rows = load_combined(function_id, history_df)
    best_idx = int(np.argmax(y))
    second_idx = int(np.argsort(y)[-2])
    best_x = X[best_idx].copy()
    second_x = X[second_idx].copy()
    latest_x = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)[-1].copy()

    config_grid = []
    for model_type in ['gp', 'mlp']:
        for beta in [0.20, 0.35, 0.50]:
            for step in [0.01, 0.02, 0.04, 0.06]:
                for grid in [0.01]:
                    config_grid.append((model_type, beta, step, grid))

    ranked = []
    for model_type, beta, step, grid in config_grid:
        score = cv_score_surrogate(X, y, model_type, seed + d)
        ranked.append((score, model_type, beta, step, grid))
    ranked.sort(key=lambda t: t[0])
    _, model_type, beta, step, grid = ranked[0]

    candidates = build_candidates(best_x, latest_x, second_x, d, step, grid)
    dist = np.sqrt(((candidates[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)) / np.sqrt(d)
    min_dist = dist.min(axis=1)
    keep = min_dist >= 0.005
    candidates = candidates[keep]
    min_dist = min_dist[keep]

    if model_type == 'gp':
        model = build_gp(seed + function_id, d)
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', category=ConvergenceWarning)
            model.fit(X, y)
        mean_pred = model.predict(candidates)
        _, std_pred = model.predict(candidates, return_std=True)
    else:
        mean_pred, std_pred = mlp_ensemble_predict(X, y, candidates, seeds=[601, 602, 603, 604])

    best_dist = np.sqrt(((candidates - best_x) ** 2).sum(axis=1)) / np.sqrt(d)
    latest_dist = np.sqrt(((candidates - latest_x) ** 2).sum(axis=1)) / np.sqrt(d)
    score = mean_pred + beta * std_pred - 0.20 * best_dist - 0.10 * latest_dist + 0.05 * min_dist
    best_candidate_idx = int(np.argmax(score))
    x_best = candidates[best_candidate_idx]

    return {
        'function': function_id,
        'd': d,
        'x': x_best,
        'formatted': format_point(x_best),
        'model_type': model_type,
        'beta': beta,
        'step': step,
        'grid': grid,
        'mean_pred': float(mean_pred[best_candidate_idx]),
        'std_pred': float(std_pred[best_candidate_idx]),
        'min_dist': float(min_dist[best_candidate_idx]),
        'best_anchor': format_point(best_x),
        'latest_anchor': format_point(latest_x),
    }


In [ ]:
results = []
for function_id in range(1, 9):
    result = suggest_week7_point(function_id, history)
    results.append(result)
    print(f'Function {function_id}: {result["formatted"]}')
    print(f'  surrogate={result["model_type"]}, beta={result["beta"]:.2f}, step={result["step"]:.2f}, grid={result["grid"]:.2f}')
    print(f'  mean={result["mean_pred"]:.6f}, std={result["std_pred"]:.6f}, min_dist={result["min_dist"]:.6f}')
    print(f'  best_anchor={result["best_anchor"]}')
    print(f'  latest_anchor={result["latest_anchor"]}')
    print()
